# 🎵 AI-Based Music Composition System - Transformer Training Notebook

This notebook guides you through training the custom Transformer Encoder-Decoder model on the MAESTRO dataset or custom MIDI files.

### Outline:
1. **Environment Setup**: Clone the repository and install all PyTorch and MIDI utility dependencies.
2. **Dataset Download**: Download a subset of the high-quality MAESTRO dataset.
3. **Tokenization & Preprocessing**: Parse MIDI files, normalize note dynamics, convert to event tokens, and segment them into sequences.
4. **Model Architecture**: Initialize the custom 18M parameter Transformer model.
5. **Model Training**: Run a PyTorch training loop using Automated Mixed Precision (AMP).
6. **Music Generation**: Generate music with the trained weights and filter through the Music Theory Constraint Engine.

In [ ]:
# Check GPU details
!nvidia-smi

In [ ]:
# Clone the repository and navigate inside
!git clone https://github.com/rohitbirdawade007/AI-Based-Music-Composition-System-Using-Transformer-Networks.git
%cd AI-Based-Music-Composition-System-Using-Transformer-Networks

In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install torch torchvision torchaudio mido midiutil pretty_midi music21 --quiet

## 1. Download MIDI Dataset
We download the MAESTRO MIDI-only dataset archive containing piano compositions.

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

dataset_dir = Path("data/maestro")
dataset_dir.mkdir(parents=True, exist_ok=True)

url = "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip"
zip_path = dataset_dir / "maestro-v3.0.0-midi.zip"

print("Downloading MAESTRO dataset (MIDI files, ~50MB)...")
urllib.request.urlretrieve(url, zip_path)

print("Extracting dataset...")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(dataset_dir)

print("Dataset extraction complete!")

## 2. Preprocessing & Tokenization
Load files, convert notes to MIDI tokens, and segment them.

In [ ]:
import sys
sys.path.insert(0, str(Path(os.getcwd())))

from src.tokenizer.music_tokenizer import MusicTokenizer
from src.data.midi_parser import MidiParser
from src.data.preprocessor import MusicPreprocessor

parser = MidiParser()
tokenizer = MusicTokenizer()
preprocessor = MusicPreprocessor()

midi_files = list(Path("data/maestro").rglob("*.mid")) + list(Path("data/maestro").rglob("*.midi"))
print(f"Found {len(midi_files)} MIDI files in dataset.")

# Limit to 30 files for a quick Colab test run
midi_files = midi_files[:30]

all_sequences = []
print("Tokenizing files...")
for idx, m_path in enumerate(midi_files):
    try:
        notes = parser.parse_file(m_path)
        if not notes:
            continue
        normalized = preprocessor.normalize_velocity(notes)
        tokens = tokenizer.encode(normalized)
        windows = preprocessor.create_sequences(tokens, seq_len=128, stride=64)
        all_sequences.extend(windows)
    except Exception as e:
        print(f"Skip {m_path.name}: {e}")

print(f"Created {len(all_sequences)} sequences of length 128!")

## 3. Create PyTorch DataLoader
Splits data into train and validation sets.

In [ ]:
import torch
from src.data.dataset import MusicDataset
from torch.utils.data import DataLoader, random_split

dataset = MusicDataset(all_sequences, dtype=torch.long)

val_size = int(len(dataset) * 0.1)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 4. Initialize Custom Transformer Model
Let's build the Transformer Encoder-Decoder from scratch.

In [ ]:
import yaml
from src.model.transformer import MusicTransformer

with open("config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_config = config["model"]
print("Hyperparameters:", model_config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MusicTransformer(
    vocab_size=model_config["vocab_size"],
    d_model=model_config["d_model"],
    nhead=model_config["nhead"],
    num_encoder_layers=model_config["num_encoder_layers"],
    num_decoder_layers=model_config["num_decoder_layers"],
    dim_feedforward=model_config["dim_feedforward"],
    dropout=model_config["dropout"],
).to(device)

print(f"Loaded model onto {device}.")
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {num_params:,}")

## 5. Model Training Loop
Train using Automated Mixed Precision (AMP).

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import time

criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding token (0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scaler = GradScaler()

epochs = 5
print(f"Training model on {device} for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    t0 = time.time()
    
    for batch_idx, (src, _) in enumerate(train_loader):
        src = src.to(device)  # (batch_size, seq_len)
        tgt_in = src[:, :-1]
        tgt_out = src[:, 1:]
        
        optimizer.zero_grad()
        with autocast(enabled=torch.cuda.is_available()):
            logits, _ = model(src_seq=src, tgt_seq=tgt_in)
            loss = criterion(logits.reshape(-1, model_config["vocab_size"]), tgt_out.reshape(-1))
            
        if torch.cuda.is_available():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
            
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Time: {time.time()-t0:.1f}s")

# Save weights
out_path = Path("output/model_checkpoint.pt")
out_path.parent.mkdir(exist_ok=True)
torch.save(model.state_dict(), out_path)
print(f"Saved weights to {out_path}")

## 6. Generate Music
Instantiate the generator and use the trained weights to compose a melody!

In [ ]:
from src.inference.generator import MusicGenerator
from src.theory.music_theory_engine import MusicTheoryEngine

generator = MusicGenerator(
    model=model,
    tokenizer=tokenizer,
    theory_engine=MusicTheoryEngine(key="C", mode="major"),
    demo_mode=False
)

result = generator.generate(
    genre="Classical",
    mood="Calm",
    key="C",
    mode="major",
    tempo=90,
    num_notes=64,
    apply_theory=True
)

print(f"Generated {result.num_notes} notes!")
print(f"Music Theory Validation Score: {result.theory_score:.3f}")
result.save_midi("model_composition.mid")
print("Saved as model_composition.mid")